# 24b — score the label-aware horizontal-flip arm (p = 0.25), against rung 21 arm A

**Control is rung 21 arm A** (`21_lr_1e4_v1`, `lr=1e-4`), **not** rung 18 — this experiment's
own README says so explicitly: "the eventual training arm will start from rung 21's exact
recipe and data." Epoch-matched by construction (RULES §6b): read rung 21 arm A's own
already-scored per-epoch series (`experiments/21-recipe-sweep/RESULTS_A_lr.csv`), never a
transcribed number.

**The one variable is the deterministic label-aware horizontal-flip policy at `p=0.25`.**
`gate_single_variable` in `_models/horizontal_flip.py` already asserted at train time that the
trained argv differs from arm A's only in `--dataset`; this notebook re-confirms that from the
checkpoint's own `args.json` rather than trusting the claim.

**The headline is the leaderboard proxy**, `mean(aggregation_ID, object_recognition_ID)`
(RULES §4b), reported beside `bucket_mean` and `margin_OOD` (RULES §4c — OOD is 50% of the
final ranking). **Pre-registered:** a win requires the proxy to RISE **and** `margin_OOD` not
to fall, matching rung 21's own pre-registration.

**🆕 targeted check** (this experiment's actual hypothesis, not inherited from rung 21): the
871 test rows the 24a audit marked `transformable` are exactly the population this
intervention should move. A generic bucket delta can hide a moved subgroup inside a flat
average — so this notebook scores that subgroup directly, paired against arm A on the SAME
qIDs, split by the three flip rules (`fixed_quadrant_class`, `object_center_quadrant`,
`all_object_positions`).


In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json, logging, os, shutil, sys, time
from pathlib import Path
import pandas as pd

# `merge_checkpoint` shells out to the bare `swift` binary. A papermill kernel does NOT
# inherit the env's bin/ on PATH, and it must be THIS interpreter's bin.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
if EXP.name != "24-geometric-aug":
    EXP = REPO / "experiments" / "24-geometric-aug"
for p in (REPO / "src", REPO / "vendor" / "orena-focus" / "src",
          EXP / "_models",
          REPO / "experiments" / "21-recipe-sweep" / "_models",
          REPO / "experiments" / "18-count-aug" / "_models",
          REPO / "experiments" / "06-vit-lora" / "_models",
          REPO / "experiments" / "02-lora-sft" / "_models"):
    if p.is_dir():
        sys.path.insert(0, str(p))

# 🔴 HF_HOME must be set BEFORE the offline flags mean anything — the judge is cached at
# /workspace/hf_cache on this pod, NOT at the default ~/.cache/huggingface (rung 18 learned
# this the hard way: an unset HF_HOME fails ~45 min in, after the merge and the full pass).
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

from frame import ledger, metrics
from frame.config import BaselineConfig
from frame.run import run_baseline
from recipe_sweep_train import RecipeSweepConfig, list_checkpoints, merge_checkpoint
import flip_audit
print("swift on PATH:", (Path(_envbin) / "swift").exists(), "| repo:", REPO)


In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) ----------
SMOKE = True          # True -> 40 questions, wiring only. Full: -p SMOKE False
EPOCH = 1             # 1 | 2 | 3 — resolved against list_checkpoints, not a step number
RUN   = "24_flip_p25_v1"

KEEP_MERGED = False   # a merged checkpoint is ~17 GB; keep only the one we ship
DATA_ROOT   = "/workspace/orena-data"

# The CONTROL is rung 21 ARM A — this experiment's own baseline (README: "the eventual
# training arm will start from rung 21's exact recipe and data"), never rung 18. Its
# per-epoch answers are archived, so every control number below is recomputed through this
# notebook's own code path rather than transcribed from RESULTS_A_lr.csv.
CONTROL_RUN = "/workspace/repo/experiments/21-recipe-sweep/runs/21_lr_1e4_v1"


In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) --
RUN_DIR = EXP / "runs" / RUN
RUN_TAG = f"ep{EPOCH}_smoke" if SMOKE else f"ep{EPOCH}_full"

# The QA parquets live in different places on the pod and on a laptop. Resolve by LOOKING,
# and fail loudly: an eval on an empty data root is the classic silent zero.
DATA_ROOT = next(
    (d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found — pull the QA parquets"

# cfg only needs to resolve THIS run's paths (ckpt_dir/merged_dir under exp_dir/runs/RUN) —
# the training engine (horizontal_flip.py) already lives in its own notebook; eval reuses
# rung 21's plain RecipeSweepConfig exactly as 21b does, and reads what was actually trained
# from args.json rather than trusting any field on this object (see next cell).
cfg = RecipeSweepConfig(exp_dir=EXP, run_name=RUN, data_root=DATA_ROOT)
CKPTS = list_checkpoints(cfg)
assert 1 <= EPOCH <= len(CKPTS), \
    f"EPOCH {EPOCH} but only {len(CKPTS)} checkpoints: {[c.name for c in CKPTS]}"
CKPT = CKPTS[EPOCH - 1]

# 🔴 The epoch-matched control is ARM A's SAME epoch, not arm A's best — the standard
# [[epoch-matched-control]] set and rung 21 itself already used against rung 18.
CTRL_RUN_DIR = Path(CONTROL_RUN)
CTRL_DIR = CTRL_RUN_DIR / f"ep{EPOCH}_full"
assert CTRL_DIR.is_dir() or SMOKE, (
    f"{CTRL_DIR} is missing — without arm A's epoch {EPOCH} there is no control for this "
    "epoch, and an unevaluated epoch is a MISSING control, not a discarded one (RULES §6b)"
)

# The recipe this run ACTUALLY trained with, read from the args.json swift wrote beside the
# checkpoints — NOT from `cfg`, whose defaults are rung 18's control values and say nothing
# about what this run trained on. "Read the artifact, never the variable" (21b's own lesson:
# its first three rows recorded lr=2e-05 for a run trained at 1e-4).
import glob as _glob
_args_json = sorted(_glob.glob(str(cfg.ckpt_dir / "*" / "args.json")))
assert _args_json, f"no args.json under {cfg.ckpt_dir} — cannot verify what was trained"
TRAINED = json.loads(Path(_args_json[-1]).read_text())
TRAINED = {k: TRAINED.get(k) for k in ("learning_rate", "lora_rank", "lora_alpha",
                                       "num_train_epochs", "seed",
                                       "per_device_train_batch_size",
                                       "gradient_accumulation_steps")}

# This experiment's single variable is the training DATA (flip policy), never the recipe —
# `gate_single_variable` already asserted this at train time (horizontal_flip.py), diffing
# the real argv against arm A's. This is that claim re-confirmed from the checkpoint itself.
_expected = {"learning_rate": 1e-4, "lora_rank": 8, "lora_alpha": 32}
for key, want in _expected.items():
    got = TRAINED[key]
    if isinstance(want, float):
        ok = got is not None and abs(got - want) < 1e-12
    else:
        ok = got == want
    if not ok:
        raise AssertionError(
            f"run {RUN!r} expects {key}={want} (arm A's recipe), but the checkpoint was "
            f"trained with {key}={got} — this run is no longer a single-variable flip A/B"
        )
if TRAINED["per_device_train_batch_size"] * TRAINED["gradient_accumulation_steps"] != 16:
    raise AssertionError(f"effective batch is not 16 in the trained run: {TRAINED}")

# The flip manifest is the export-time proof that this run's data really differs from the
# control — a run whose manifest never flipped a row would be arm A wearing a new name.
_manifest_path = cfg.run_dir / "flip_manifest.csv"
assert _manifest_path.exists(), f"no flip manifest at {_manifest_path} — was `stage='export'` run?"
_manifest = pd.read_csv(_manifest_path)
assert _manifest["flipped"].any(), f"{_manifest_path} flipped zero rows — this is arm A, not a flip arm"

print(f"SMOKE   {SMOKE}")
print(f"epoch   {EPOCH}/{len(CKPTS)} -> {CKPT.name}")
print(f"control {CTRL_DIR}   (rung 21 arm A, same epoch)")
print(f"tag     {RUN_TAG}")
print(f"trained {TRAINED}")
print(f"manifest: {_manifest['flipped'].sum()}/{len(_manifest)} rows flipped, "
      f"by rule: {_manifest.groupby('flipped')['rule'].value_counts().to_dict()}")


In [ ]:
# --- PRE-FLIGHT: the LLM judge must resolve offline, BEFORE anything expensive ---
# `run_baseline` loads the judge only AFTER the 17 GB merge and the full inference pass, so a
# missing judge cache fails ~45 minutes in with everything already paid for. This is that
# failure, hoisted to the front and made cheap: the tokenizer alone proves the cache resolves
# under the offline flags. RAISES (RULES §7).
from transformers import AutoTokenizer

_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}. The judge is cached at /workspace/hf_cache "
        "on this pod, not at the default ~/.cache/huggingface. Fix the env — do NOT disable "
        "the offline flags, the whole deployment story is offline."
    ) from exc
print(f"OK judge gate: {_judge} resolves from {os.environ.get('HF_HOME')}")


In [ ]:
# --- merge the adapter (per-epoch, namespaced so merges never overwrite) ---------
merged = cfg.merged_dir / CKPT.name
if merged.is_dir() and any(merged.iterdir()):
    print(f"OK    merged already present -> {merged}")
    MERGED_HERE = False
else:
    t0 = time.perf_counter()
    merged = merge_checkpoint(cfg, CKPT)
    MERGED_HERE = True
    print(f"      merged in {time.perf_counter() - t0:.0f}s -> {merged}")
merged


In [ ]:
# --- eval -----------------------------------------------------------------------
t0 = time.perf_counter()
try:
    cfg_eval = BaselineConfig(
        data_root=DATA_ROOT, model_path=merged, out_dir=RUN_DIR, run_name=RUN_TAG,
        max_pixels=1280 * 720, seed=42, n_eval=40 if SMOKE else None,
    )
    # The inference path must stay rung 06's / rung 21's EXACTLY. This rung's variable is
    # the training DATA; a post-processor or a second sample here would be a second
    # variable and the delta would stop being attributable to the flip policy alone.
    assert cfg_eval.answer_postprocess is None, "answer_postprocess must stay None"
    assert cfg_eval.n_samples == 1 and cfg_eval.enhance is None and cfg_eval.aux_view is None
    report = run_baseline(cfg_eval)
    print(f"eval done in {(time.perf_counter() - t0) / 60:.1f} min")
finally:
    if MERGED_HERE and not KEEP_MERGED and Path(merged).is_dir():
        shutil.rmtree(merged, ignore_errors=True)
        print(f"reclaimed ~17 GB -> removed {merged}")


In [ ]:
# --- canonical scoring + gates (all RAISE) --------------------------------------
gold = ledger.gold_from_frame_parquets(DATA_ROOT)
res = pd.read_csv(RUN_DIR / RUN_TAG / "results.csv")

missing = set(res["qID"]) - set(gold.dropna(subset=["answer"])["qID"])
assert not missing, f"GATE 0 — {len(missing)} qIDs without gold; every margin would be inflated"

metrics.assert_no_dup_qid(res)
metrics.assert_ood_from_qid(res)
metrics.assert_all_rows_grouped(res)
strat = metrics.stratified_report(res, gold=gold)
metrics.assert_floors_vs_eval_set(strat)

# The mode gate, on the ARTIFACT rather than the variable: `-p SMOKE False` failing to take
# effect is otherwise indistinguishable from a successful full run.
assert (len(res) < 1000) == SMOKE, (
    f"MODE GATE FAILED: SMOKE={SMOKE} but the eval scored {len(res)} rows")
if not SMOKE:
    assert len(res) == 6252, f"expected the full eval set, got {len(res)} rows"

print("gates OK | bucket_mean:", round(strat["bucket_mean"], 4),
      "| margin_OOD:", round(strat["margin_OOD"], 4))


In [ ]:
# --- 🎯 THE HEADLINE: the leaderboard proxy, epoch-matched against rung 21 arm A -
# mean(aggregation_ID, object_recognition_ID) — the quantity the platform actually scores on
# the pre-eval, where only those two buckets populate and both are ID (RULES §4b). It is NOT
# `bucket_mean`, and the two must never be quoted against each other.
def _proxy(report: dict) -> dict:
    # 🔴 A missing bucket is expected in SMOKE (a 40-row sample need not contain any
    # aggregation x ID question) and is a FINDING in a full run — returns NaN here, and the
    # gate below raises only when the whole eval set was scored.
    bb = pd.DataFrame(report["by_bucket"])
    idc = bb[bb.distribution == "ID"].set_index("capability_group")
    out = {}
    for key, name in (("aggregation_ID", "aggregation"),
                      ("object_recognition_ID", "object_recognition")):
        out[key] = float(idc.loc[name, "accuracy"]) if name in idc.index else float("nan")
    out["proxy"] = (out["aggregation_ID"] + out["object_recognition_ID"]) / 2
    return out

arm_p = _proxy(strat)
if not SMOKE and any(pd.isna(v) for v in arm_p.values()):
    raise AssertionError(
        f"a scored bucket is missing from the full eval: {arm_p} — the leaderboard proxy is "
        "the mean of exactly these two, so a missing one is not a smaller number, it is no "
        "number at all"
    )

# The control is recomputed from arm A's OWN archived answers through this same function —
# a transcribed number cannot be re-derived, and this one can.
ctrl_strat = None
ctrl_res = None
if CTRL_DIR.is_dir() and (CTRL_DIR / "results.csv").exists():
    ctrl_res = pd.read_csv(CTRL_DIR / "results.csv")
    # Independent-review finding: the arm's own results.csv is checked for duplicate
    # qIDs (cell 7) but the control's never was -- a dup here would silently
    # cartesian-multiply that row in cell 13's merge rather than raise.
    metrics.assert_no_dup_qid(ctrl_res)
    ctrl_strat = metrics.stratified_report(ctrl_res, gold=gold)
elif (CTRL_DIR / "stratified.json").exists():
    ctrl_strat = json.loads((CTRL_DIR / "stratified.json").read_text())
    ctrl_res = None

if ctrl_strat is None:
    print(f"⚠️  no control artifacts under {CTRL_DIR} — deltas below are NOT computed")
    ctrl_p = {k: float("nan") for k in arm_p}
else:
    ctrl_p = _proxy(ctrl_strat)

print(pd.DataFrame([
    {"cell": k, f"armA_ep{EPOCH}": round(ctrl_p[k], 4), f"flip_p25_ep{EPOCH}": round(arm_p[k], 4),
     "delta": round(arm_p[k] - ctrl_p[k], 4)}
    for k in ("aggregation_ID", "object_recognition_ID", "proxy")
]).to_string(index=False))


In [ ]:
# --- class-balanced F1 on `fo_class` — the metric the headline hides ------------
# More optimisation distance (or a data change like this one) is most likely to do its
# damage or its good in the TAIL, and exact-set accuracy cannot see the tail.
preds = metrics.predictions_frame(RUN_DIR / RUN_TAG)
f1 = metrics.class_f1_report(preds, gold, results_df=res, n_boot=0 if SMOKE else 2000)

ctrl_f1 = None
if CTRL_DIR.is_dir() and (CTRL_DIR / "predictions.json").exists():
    ctrl_preds = metrics.predictions_frame(CTRL_DIR)
    ctrl_f1 = metrics.class_f1_report(ctrl_preds, gold, n_boot=0)

rows = []
for cell in ("pooled", "ID", "OOD"):
    b = f1[cell]
    a = ctrl_f1[cell] if ctrl_f1 else {"macro_f1": float("nan"), "exact_set_acc": float("nan")}
    rows.append({
        "cell": cell, "n": b["n"], "illegal": b["n_illegal"],
        "macro_f1_armA": round(a["macro_f1"], 4), "macro_f1_flip": round(b["macro_f1"], 4),
        "d_macro": round(b["macro_f1"] - a["macro_f1"], 4),
        "exact_armA": round(a["exact_set_acc"], 4), "exact_flip": round(b["exact_set_acc"], 4),
        "ci_flip": f"[{b['ci_low']:.3f}, {b['ci_high']:.3f}]",
    })
f1_df = pd.DataFrame(rows)
print(f1_df.to_string(index=False))

_per = pd.DataFrame(f1["ID"]["per_class"]).T
print("\n--- per class, ID (the tail is the point) ---")
if _per.empty:
    print("   (no fo_class x ID rows in this sample — expected in SMOKE only)")
else:
    print(_per[["n_gold", "recall", "precision", "f1"]]
          .sort_values("n_gold", ascending=False).round(3).to_string())


In [ ]:
# --- count discrimination on the `Clips` template -------------------------------
# ⚠️ A rank correlation is a property of the SLICE, never of the model (RULES §13b): quote the
# template and n every time. And rank does NOT cash into score (§13c).
rank = metrics.count_rank_report(preds, gold)
ref_rank = (metrics.count_rank_report(metrics.predictions_frame(CTRL_DIR), gold)
            if CTRL_DIR.is_dir() and (CTRL_DIR / "predictions.json").exists() else None)

rows = []
for cell in ("pooled", "ID", "OOD"):
    b = rank[cell]
    a = ref_rank[cell] if ref_rank else {"r": float("nan"), "bias": float("nan"),
                                         "exact": float("nan")}
    rows.append({
        "cell": cell, "n": b["n"],
        f"r_armA_ep{EPOCH}": round(a["r"], 4), "r_flip": round(b["r"], 4),
        "delta_r": round(b["r"] - a["r"], 4),
        "ci_flip": f"[{b['ci_low']:.3f}, {b['ci_high']:.3f}]",
        "bias_armA": round(a["bias"], 3), "bias_flip": round(b["bias"], 3),
        "unreadable_flip": b["n_unreadable"],
    })
rank_df = pd.DataFrame(rows)
print(rank_df.to_string(index=False))


## 🆕 the actual hypothesis: the 871 `transformable` test rows, paired against arm A

Every other cell above reads a generic delta that this flip policy was never specifically
aimed at. The 24a audit already named the exact 871 test-set rows (`heico`+`lapchole`, 3
rules) where a horizontal reflection changes a spatial question or answer. If the flip
augmentation teaches anything about camera-relative left/right, it should show up **here**,
paired on the SAME qIDs against arm A — not diluted across the other 5,381 rows that carry no
spatial signal either way.

The audit is recomputed here (not read from a cached CSV) against the pod's own
`{data_root}/<dataset>/data/frame/test.parquet` — the same files `gold_from_frame_parquets`
just read — so this cell cannot drift from what the eval actually scored, and the row count is
asserted against the committed README's audit result (871) as a drift check on the audit code
itself.


In [ ]:
# --- recompute the 24a test-split audit against THIS pod's parquets -------------
import glob as _glob2

_audit_frames = []
for f in sorted(_glob2.glob(str(Path(DATA_ROOT) / "*" / "data" / "frame" / "test.parquet"))):
    ds = Path(f).parents[2].name
    a = flip_audit.audit_parquet(f, split="test", dataset=ds)
    a["qID"] = ds + "__" + a["id"].astype(str)
    _audit_frames.append(a)
test_audit = pd.concat(_audit_frames, ignore_index=True)

transformable = test_audit[test_audit.disposition == "transformable"]
if not SMOKE:
    assert len(transformable) == 871, (
        f"expected 871 transformable test rows per the committed README, got {len(transformable)} "
        "— the audit rules drifted from what was reported; do not read the cells below until "
        "that is reconciled"
    )
print(f"transformable test rows: {len(transformable)} "
      f"(SMOKE eval only scored {len(res)} of the full 6252, so most may be absent below)")
print(transformable["rule"].value_counts().to_string())


In [ ]:
# --- targeted paired accuracy on the transformable subset, vs arm A same epoch --
if ctrl_res is None:
    print(f"⚠️  no control results.csv under {CTRL_DIR} — targeted deltas NOT computed")
    targeted_rows = []
else:
    sub_arm = (res[res.qID.isin(transformable.qID)][["qID", "video", "correctness"]]
               .rename(columns={"correctness": "correct_b"}))
    sub_ctrl = (ctrl_res[["qID", "correctness"]]
                .rename(columns={"correctness": "correct_a"}))
    merged_sub = sub_arm.merge(sub_ctrl, on="qID", how="inner")
    if not SMOKE and len(merged_sub) != len(sub_arm):
        raise AssertionError(
            f"control is missing {len(sub_arm) - len(merged_sub)} of the transformable qIDs "
            "this arm scored — the two runs are not on the same question set"
        )

    targeted_rows = []
    for label, qids in (("ALL_transformable", transformable.qID),
                        *((f"rule={r}", transformable.loc[transformable.rule == r, "qID"])
                          for r in transformable["rule"].unique())):
        sub = merged_sub[merged_sub.qID.isin(qids)]
        # unlike class_f1_report/count_rank_report, paired_delta_ci has no n_boot=0
        # guard -- it always calls np.percentile on the boot array, which raises
        # IndexError on an EMPTY array (size 0) rather than returning NaN. A small
        # nonzero n_boot keeps SMOKE fast without hitting that.
        ci = metrics.paired_delta_ci(sub, correct_a="correct_a", correct_b="correct_b",
                                     n_boot=200 if SMOKE else 2000)
        # paired_delta_ci returns delta/ci_low/ci_high/n/n_videos/wins_b/wins_a -- it does
        # NOT return `excludes_0` (that column in RESULTS_paired_ci.csv was computed by a
        # one-off script, not this function). Derive it here; NaN bounds (n_videos < 3,
        # or an empty slice) must stay unknown, not silently read as True.
        import math as _math
        _lo, _hi = ci["ci_low"], ci["ci_high"]
        ci["excludes_0"] = (
            None if (_math.isnan(_lo) or _math.isnan(_hi)) else bool(_lo > 0 or _hi < 0)
        )
        targeted_rows.append({"cell": label, **ci})
    targeted_df = pd.DataFrame(targeted_rows)
    print(targeted_df.to_string(index=False))
    if not SMOKE and not targeted_df.empty:
        print("\n⚠️  a delta with a CI that does not exclude zero is not a result (RULES / "
              "[[epoch-matched-control]]'s own standard) — read `excludes_0`, not the sign.")


In [ ]:
# --- the run's row + the PRE-REGISTERED verdict ---------------------------------
bf = pd.DataFrame(strat["by_format"])
num = bf[bf.answer_format == "number"].set_index("distribution")
ctrl_margin_ood = float(ctrl_strat["margin_OOD"]) if ctrl_strat else float("nan")

_targeted_all = next((r for r in targeted_rows if r["cell"] == "ALL_transformable"), None)

row = {
    "run": RUN, "epoch": EPOCH, "checkpoint": CKPT.name,
    "baseline_run": CTRL_RUN_DIR.name,
    "flip_probability": 0.25,
    "n_flipped_train_rows": int(_manifest["flipped"].sum()),
    "lr": TRAINED["learning_rate"], "lora_rank": TRAINED["lora_rank"],
    "lora_alpha": TRAINED["lora_alpha"],
    "proxy_leaderboard": arm_p["proxy"],
    "aggregation_ID": arm_p["aggregation_ID"],
    "object_recognition_ID": arm_p["object_recognition_ID"],
    "bucket_mean": strat["bucket_mean"],
    "acc_ID": strat["acc_ID"], "acc_OOD": strat["acc_OOD"],
    "margin_ID": strat["margin_ID"], "margin_OOD": strat["margin_OOD"],
    "number_margin_ID": num.loc["ID", "margin"] if "ID" in num.index else float("nan"),
    "number_margin_OOD": num.loc["OOD", "margin"] if "OOD" in num.index else float("nan"),
    "fo_class_macro_f1_ID": f1["ID"]["macro_f1"],
    "fo_class_macro_f1_OOD": f1["OOD"]["macro_f1"],
    "clips_r_pooled": rank["pooled"]["r"], "clips_bias": rank["pooled"]["bias"],
    "transformable_delta": _targeted_all["delta"] if _targeted_all else float("nan"),
    "transformable_ci_low": _targeted_all["ci_low"] if _targeted_all else float("nan"),
    "transformable_ci_high": _targeted_all["ci_high"] if _targeted_all else float("nan"),
    "transformable_excludes_0": _targeted_all["excludes_0"] if _targeted_all else None,
    "d_proxy_vs_armA_same_epoch": arm_p["proxy"] - ctrl_p["proxy"],
    "d_margin_OOD_vs_armA_same_epoch": strat["margin_OOD"] - ctrl_margin_ood,
}
print(pd.DataFrame([row]).T.to_string(header=False))

_proxy_up = row["d_proxy_vs_armA_same_epoch"] > 0
_ood_held = row["d_margin_OOD_vs_armA_same_epoch"] >= 0
print(f"\nPRE-REGISTERED READ (vs rung 21 arm A epoch {EPOCH})")
print(f"   leaderboard proxy rises : {_proxy_up}  "
      f"({row['d_proxy_vs_armA_same_epoch']:+.4f})")
print(f"   margin_OOD does not fall: {_ood_held}  "
      f"({row['d_margin_OOD_vs_armA_same_epoch']:+.4f})")
print(f"   -> {'WIN' if (_proxy_up and _ood_held) else 'NOT A WIN'}")
print(f"\nTARGETED READ (the 871 transformable rows, paired, vs arm A)")
if _targeted_all:
    print(f"   delta={_targeted_all['delta']:+.4f}  "
          f"CI=[{_targeted_all['ci_low']:.4f}, {_targeted_all['ci_high']:.4f}]  "
          f"excludes_0={_targeted_all['excludes_0']}")
else:
    print("   not computed (see targeted cell above)")
print("\n⚠️  There is NO seed-variance estimate in this project. A delta smaller than the "
      "paired video-clustered CI is not a result — quote the CI or do not quote the delta.")
print("A faithful negative is a real result and is recorded as one.")


In [ ]:
# --- persist: RESULTS.csv + the ledger (full runs only) -------------------------
if not SMOKE:
    # 🔴 One CSV PER ARM, not one shared file (21b's own lesson): two pods can share this
    # network volume, and read-concat-write is not atomic.
    out = EXP / "RESULTS_flip_p25.csv"
    df = pd.concat([pd.read_csv(out), pd.DataFrame([row])], ignore_index=True) if out.exists() \
        else pd.DataFrame([row])
    df.to_csv(out, index=False)
    f1_df.to_csv(EXP / f"RESULTS_class_f1_flip_p25_ep{EPOCH}.csv", index=False)
    rank_df.to_csv(EXP / f"RESULTS_rank_flip_p25_ep{EPOCH}.csv", index=False)
    if targeted_rows:
        pd.DataFrame(targeted_rows).to_csv(
            EXP / f"RESULTS_transformable_flip_p25_ep{EPOCH}.csv", index=False)
    ledger.register_run(RUN_DIR / RUN_TAG, strat, experiment="24-geometric-aug",
                        run=f"{RUN}__{RUN_TAG}",
                        model=f"24 flip_p25 epoch {EPOCH} ({CKPT.name})",
                        date="2026-07-31")
    print("wrote", out)
else:
    print("SMOKE — nothing persisted to RESULTS.csv or the ledger")


In [ ]:
# --- eyeball: what did it actually SAY? (user rule: examples on every run) -------
g = gold.copy()
g["template"] = g["question"].map(metrics.template_of)
clips = g[g.template == "How many Clips appear in this frame? Please provide a number."]
look = clips.merge(preds, on="qID").head(400)
look["gold_n"] = look["answer"].map(metrics.read_count)
look["pred_n"] = look["prediction"].map(metrics.read_count)
look["err"] = look["pred_n"] - look["gold_n"]

print("--- predicted-vs-gold crosstab (Clips) ---")
print(pd.crosstab(look["gold_n"], look["pred_n"]).to_string())
print("\n--- the worst counting misses ---")
print(look.reindex(look["err"].abs().sort_values(ascending=False).index)
          .head(12)[["qID", "gold_n", "pred_n", "prediction"]].to_string(index=False))

fo = res[res.answer_format == "fo_class"].merge(preds, on="qID")
wrong = fo[fo.correctness == 0].head(10)
print("\n--- fo_class misses (identity, not cardinality, is the usual failure) ---")
print(wrong[["qID", "prediction"]].to_string(index=False))

# 🆕 the quadrant/spatial predictions THIS experiment targets — every miss here is a miss on
# exactly the population the flip policy was meant to help.
spatial = transformable.merge(preds, on="qID", how="inner").merge(
    res[["qID", "correctness"]], on="qID", how="inner")
print("\n--- transformable-subset misses, by rule (the targeted population) ---")
if spatial.empty:
    print("   (none scored in this sample — expected in SMOKE)")
else:
    sp_wrong = spatial[spatial.correctness == 0]
    print(sp_wrong.groupby("rule").size().to_string())
    print(sp_wrong[["qID", "rule", "answer", "prediction"]].head(12).to_string(index=False))


## After this epoch — and after all three

1. **Read the proxy per epoch**, epoch-matched against rung 21 arm A. A win needs it to rise
   *and* `margin_OOD` not to fall, at the same epoch.
2. **Read the targeted transformable-subset delta separately from the generic proxy.** A
   generic win with a flat/negative targeted delta means the gain (if any) is not coming from
   the mechanism this experiment tests — that is a finding, not a discard.
3. **Select by `acc_OOD`, per epoch** — never last-epoch-by-default (RULES §6). OOD is 50% of
   the final ranking.
4. **If this is a faithful negative or a wash**, that closes `p=0.25` as a lever and is a real
   result — record it in `context/decisions/` before touching `p=0.50` (24c). The README
   already pre-registers 24c as a SEPARATE reported rung, not a hyperparameter sweep folded
   into this one.
5. **If this is a win**, only then does 24c (`p=0.50`) become worth the GPU-hours — per the
   README's registered probability sequence.
6. Update `experiments/24-geometric-aug/README.md`'s ladder table, `context/24-geometric-aug/
   CONTEXT.md` Results/Next, and `context/NOW.md`. Regenerate the ledger
   (`python -m frame.ledger` or the notebook equivalent) so Tier 1–3 pick up the new row.
